In [ ]:
#INSTRUCCIONES
#Analicen los eventos registrados en todo el mundo a partir del proyecto GDELT https://blog.gdeltproject.org/gdelt-2-0-our-global-world-in-realtime/

#Recursos:

#Lista de URLs con actualizaciones cada 15 minutos https://data.gdeltproject.org/gdeltv2/masterfilelist.txt .
#URL oficial del libro de códigos de GDELT 2.0 https://data.gdeltproject.org/documentation/GDELT-Event_Codebook-V2.0.pdf
#URL de la documentación en texto/HTML donde se enuncian los campos https://gdeltproject.org/data/documentation/CAMEO.Manual.1.1b3.pdf

#Actividades:

#1) Accedan a los datos de la forma más eficiente posible. Fundamenten sus decisiones.

#2) Calculen las siguientes medidas diarias desde 2 Sep 2026:

#Cantidad de eventos en el día
#Promedio de artículos por evento en el día
#Promedio de fuentes por evento en el día

#3) Presenten los resultados como un DataFrame de Pandas.

#Formato de entrega: notebook con el nombre "TP1" dentro de su repositorio Github. En esta actividad deben pegar la URL con el acceso a la notebook.

In [ ]:
# Nota sobre el desarrollo de este TP:
# Para el armado de este notebook se utilizó Inteligencia Artificial (Gemini) como colaboradora técnica.
# Mi rol consistió en definir la idea y brindar el flujo metodológico con las herramientas vistas en clase
# La IA asistió en la redacción del código, aclaración de dudas y errores.

In [1]:
# 1 -------- IMPORTS
# Cargar módulos de Python necesarios para el TP

import io
import os
import re
import zipfile
from datetime import datetime
import pandas as pd
import psutil
import pyarrow as pa
import pyarrow.csv as csv
import requests
from IPython.display import display

In [ ]:
# 2 -  OPCIONAL!!!!!!!!!!!

# INSPECCIÓN PRIMARIA DE LOS DATOS

# Inspeccionar y visualizar los datos, con una muestra reducida.
# Descargar un único lote de 15 minutos para validar la conexión y explorar el formato de GDELT


# Acceder a URL de un lote de 15 minutos del 02/Sep/2026 (00:00:00 UTC)
url_sample = "http://data.gdeltproject.org/gdeltv2/20260902000000.export.CSV.zip"

print("Descargando lote de prueba...")
r = requests.get(url_sample)

# Descomprimir en memoria y leer con Pandas
with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    csv_filename = z.namelist()[0]
    with z.open(csv_filename) as f:
        # Nota: GDELT no incluye cabeceras en el CSV, usa tabulaciones (\t) como separador
        df_sample = pd.read_csv(f, sep="\t", header=None, usecols=[0, 32, 33])

# Dado que los archivos CSV de GDELT no incluyen cabeceras (no tienen los nombres de las columnas en la primera fila),
# para interpretar qué significa cada número de columna recurri a su especificación técnica en https://data.gdeltproject.org/documentation/GDELT-Event_Codebook-V2.0.pdf

# Seleccione: GlobalEventID (Índice 0)
#             NumArticles (Índice 33)
#             NumSources (Índice 32)
# Les coloco sus respectivos nombres

df_sample.columns = ["GlobalEventID", "NumSources", "NumArticles"]

print(f"Lote cargado exitosamente. Filas: {len(df_sample)}, Columnas: {len(df_sample.columns)}")
print() # Espacio en blanco

# Ver las primeras 5 filas de la información
display(df_sample.head())

# Para inspeccionar solo las columnas de mi interés y calcular las métricas de ese batch de información reducida:

print()
print("--- RESUMEN DEL BATCH (15 MIN) ---")
print(f"Total de eventos en este lote: {len(df_sample)}")
print(f"Promedio de artículos por evento: {df_sample['NumArticles'].mean():.2f}")
print(f"Promedio de fuentes por evento: {df_sample['NumSources'].mean():.2f}")

In [2]:
# 3 -------- INGESTA DE GRAN CANTIDAD DE DATOS

# Conexion con la lista maestra de GDELT, procesa mediante lectura por líneas (streaming).
# Utiliza una expresión regular (regex) y filtrado por objetos datetime para identificar y extraer las URLs de los archivos .export.CSV.zip
# generados a partir del 2 de septiembre de 2026, agrupándolos por fecha en un diccionario.


URL_MASTERFILELIST = "https://data.gdeltproject.org/gdeltv2/masterfilelist.txt"
FECHA_INICIO = datetime(2026, 9, 2)
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

print("1. Obteniendo links desde masterfilelist.txt...")
response = requests.get(URL_MASTERFILELIST, headers=headers, stream=True)

urls_por_dia = {}
regex_export = re.compile(r"http://data\.gdeltproject\.org/gdeltv2/(\d{14})\.export\.CSV\.zip")

# Lectura línea a línea (Streaming)
for line in response.iter_lines():
    if line:
        decoded_line = line.decode("utf-8")
        match = regex_export.search(decoded_line)
        if match:
            timestamp_str = match.group(1)
            fecha_archivo = datetime.strptime(timestamp_str, "%Y%m%d%H%M%S")

            if fecha_archivo >= FECHA_INICIO:
                fecha_str = fecha_archivo.strftime("%Y-%m-%d")
                link_datos = decoded_line.split()[-1]
                urls_por_dia.setdefault(fecha_str, []).append(link_datos)

print(f"   -> {sum(len(v) for v in urls_por_dia.values())} links encontrados a partir del 02/Sep/2026.")

1. Obteniendo links desde masterfilelist.txt...
   -> 866 links encontrados a partir del 02/Sep/2026.


In [3]:
# 4 -------- PROCESAMIENTO POR BLOQUES

# Procesa todos los links agrupados por día sin saturar la RAM.
# Descomprime cada lote en memoria, utiliza PyArrow para hacer una lectura eficiente desde C++ con manejo seguro de flotantes y nulos (float64).
# Convierte el fragmento a Pandas para sumar los artículos y fuentes de forma escalar.
# Monitorea el consumo de memoria con psutil lote a lote.
# Devuelve el DataFrame consolidado con la cantidad total de eventos y las promedios diarios requeridos.

proceso = psutil.Process(os.getpid())

column_names = ["GlobalEventID" if i == 0 else "NumArticles" if i == 33 else "NumSources" if i == 32 else f"col_{i}" for i in range(61)]

# Configuración de PyArrow
pyarrow_read_options = csv.ReadOptions(column_names=column_names)
pyarrow_parse_options = csv.ParseOptions(delimiter="\t")
pyarrow_convert_options = csv.ConvertOptions(
    column_types={
        "GlobalEventID": pa.int64(),
        "NumArticles": pa.float32(),
        "NumSources": pa.float32(),
    },
    include_columns=["GlobalEventID", "NumArticles", "NumSources"],
)

print("\n" + "=" * 60)
print("AGREGACIÓN POR BLOQUES EN PYARROW (MONITOREO DE RAM)")
print("=" * 60)

resultados_diarios = []

for fecha, links_del_dia in urls_por_dia.items():
    print(f"\n---> PROCESANDO DÍA: {fecha} ({len(links_del_dia)} lotes)")

    total_eventos = 0
    total_articulos = 0
    total_fuentes = 0

    for batch_idx, link in enumerate(links_del_dia):
        try:
            r = requests.get(link, headers=headers, timeout=10)

            with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                csv_filename = z.namelist()[0]
                csv_bytes = z.read(csv_filename)

                # Lectura por PyArrow Buffer (Argumentos corregidos)
                reader = pa.BufferReader(csv_bytes)
                table = csv.read_csv(
                    reader,
                    read_options=pyarrow_read_options,
                    parse_options=pyarrow_parse_options,
                    convert_options=pyarrow_convert_options,
                )

                # Bloque parcial a Pandas
                batch_df = table.to_pandas()
                batch_df["NumArticles"] = batch_df["NumArticles"].fillna(0).astype("int32")
                batch_df["NumSources"] = batch_df["NumSources"].fillna(0).astype("int32")
                eventos_bloque = len(batch_df)
                articulos_bloque = batch_df["NumArticles"].sum()
                fuentes_bloque = batch_df["NumSources"].sum()

                total_eventos += eventos_bloque
                total_articulos += articulos_bloque
                total_fuentes += fuentes_bloque

                if batch_idx < 3 or (batch_idx + 1) % 20 == 0:
                    ram_mb = proceso.memory_info().rss / (1024**2)
                    print(
                        f"  Bloque {batch_idx+1}/{len(links_del_dia)}: {eventos_bloque:,} eventos procesados | RAM actual: {ram_mb:.2f} MB"
                    )

        except Exception as e:
            print(f"  Error en bloque {batch_idx+1}: {e}")

    prom_art = total_articulos / total_eventos if total_eventos > 0 else 0
    prom_fue = total_fuentes / total_eventos if total_eventos > 0 else 0

    resultados_diarios.append(
        {
            "Fecha": fecha,
            "Cantidad de eventos": total_eventos,
            "Promedio artículos/evento": round(prom_art, 2),
            "Promedio fuentes/evento": round(prom_fue, 2),
        }
    )

# METRICAS FINALES
df_final = pd.DataFrame(resultados_diarios)
print("\n" + "=" * 60)
print("RESULTADOS FINALES POR DÍA")
print("=" * 60)
display(df_final)


AGREGACIÓN POR BLOQUES EN PYARROW (MONITOREO DE RAM)

---> PROCESANDO DÍA: 2026-09-02 (96 lotes)
  Bloque 1/96: 2,408 eventos procesados | RAM actual: 183.18 MB
  Bloque 2/96: 1,646 eventos procesados | RAM actual: 184.39 MB
  Bloque 3/96: 1,507 eventos procesados | RAM actual: 184.94 MB
  Bloque 20/96: 1,066 eventos procesados | RAM actual: 187.57 MB
  Bloque 40/96: 1,356 eventos procesados | RAM actual: 188.58 MB
  Bloque 60/96: 1,431 eventos procesados | RAM actual: 189.62 MB
  Bloque 80/96: 1,586 eventos procesados | RAM actual: 189.76 MB

---> PROCESANDO DÍA: 2026-09-03 (96 lotes)
  Bloque 1/96: 2,222 eventos procesados | RAM actual: 190.54 MB
  Bloque 2/96: 1,140 eventos procesados | RAM actual: 190.54 MB
  Bloque 3/96: 1,556 eventos procesados | RAM actual: 190.54 MB
  Bloque 20/96: 1,007 eventos procesados | RAM actual: 190.59 MB
  Bloque 40/96: 962 eventos procesados | RAM actual: 190.80 MB
  Bloque 60/96: 1,396 eventos procesados | RAM actual: 192.17 MB
  Bloque 80/96: 965 e

,Fecha,Cantidad de eventos,Promedio artículos/evento,Promedio fuentes/evento
0,2026-09-02,120105,4.64,1.05
1,2026-09-03,117020,4.63,1.05
2,2026-09-04,107037,4.69,1.05
3,2026-09-05,66878,4.64,1.04
4,2026-09-06,60454,4.57,1.04
5,2026-09-07,86043,4.68,1.04
6,2026-09-08,114917,4.60,1.05
7,2026-09-09,119065,4.63,1.04
8,2026-09-10,117648,4.60,1.04
9,2026-09-11,3175,5.52,1.20
